# IIMT Native: CSV → одна / две руки
Без ROS. `pip install "iimt-native[notebook]"` из проверенного wheel. По умолчанию используются **внешние TCP mock-процессы**, не реальные роботы.

CSV содержит время `t` (секунды) и `left_q1_rad…left_q6_rad`, `right_q1_rad…right_q6_rad`. Частота CSV произвольная, допускаются неравномерные timestamps; без времени задайте CSV_SOURCE_HZ. Производные для отправки не создаются: вычисления ниже нужны только для preview. CSV_RESAMPLE_HZ=None и TIME_SCALE=1 сохраняют исходные данные. Подвод — отдельная ячейка с RRTConnect. Графики выполнения используют **UTC**; offline-preview — относительное время, ошибка = фактическое − заданное. Timestamp измерения — получение ответа на ПК, не время энкодера. Упреждение изменяет отправку, не ось графика.

На настоящем стенде проверьте URDF/основания/workspace, аварийный останов и допустимость траектории. Mock не доказывает динамику или безопасность железа.

In [ ]:
%matplotlib inline
from pathlib import Path
import time
import numpy as np
from iimt import Arm, Robot, Trajectory, ci05_scene
from iimt.mock import MockRobot
from iimt.plotting import plot_execution, plot_trajectory, trajectory_limit_report

DUAL = False
USE_MOCK = True
VIEWER = True  # True — отдельное 3D окно; нужен графический рабочий стол.
RUN_APPROACH = True
RUN_CSV = True
LEFT_IP, RIGHT_IP = "127.0.0.1","127.0.0.1"#"192.168.50.95", "192.168.50.163"
LEFT_PORT = RIGHT_PORT = 29999
COMMAND_HZ, READ_HZ = 125., 125.
ADVANCE_MS = [320., 320.]  # независимые калиброванные значения, не универсальные 330 мс.
START_DELAY_S = 3.0    # timestamp сообщения создаётся ПОСЛЕ collision preflight.
# CSV: частота точек независима от COMMAND_HZ / READ_HZ.
CSV_TIME_COLUMN = "t"  # Секунды; None, если столбца времени нет.
CSV_ZERO_START = True  # Вычесть t[0], чтобы не добавлялся неявный подвод до CSV.
CSV_SOURCE_HZ = None   # Например 100., только при CSV_TIME_COLUMN=None.
CSV_RESAMPLE_HZ = None # None — все исходные точки; 30. — сетка ДО замедления.
CSV_RESAMPLE_MODE = "interpolate"  # "select" — только исходные строки и timestamps.
TIME_SCALE = 1.0       # 3.0 — в 3 раза медленнее, углы не меняются.

# Лимиты в SDK-порядке J1…J6: rad, rad/s, rad/s², rad/команду.
# Позиции и скорости дополнительно ограничены URDF; большие значения не обходят URDF.
LEFT_LIMITS = dict(
    min_position=[-6.2832, -2.6167, -2.8783, -3.0528, -3.0528, -6.2832],
    max_position=[ 6.2832,  2.6167,  2.8783,  3.0528,  3.0528,  6.2832],
    max_velocity=[3.1415] * 6,
    max_acceleration=[1.875, 2.5, 3.75, 3.125, 3.75, 3.75],
    max_step=[0.02] * 6,
)
RIGHT_LIMITS = {key: values.copy() for key, values in LEFT_LIMITS.items()}
# Например: LEFT_LIMITS["max_velocity"][0] = 0.5
#           RIGHT_LIMITS["max_acceleration"] = [1.0] * 6

WORKSPACE_MARGINS = (.5, .5, .5, .5)  # X−, X+, Y−, Y+, метры.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/trajectory_both_TimeGrid_30Hz.csv").exists())
CSV_PATH = ROOT / "data/trajectory_both_TimeGrid_30Hz.csv"
OUTPUT = ROOT / "results" / time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
scene = ci05_scene(dual=DUAL, margins=WORKSPACE_MARGINS)
raw_trajectory = Trajectory.from_csv(
    CSV_PATH, scene.joint_names, arm="both" if DUAL else "left",
    time_column=CSV_TIME_COLUMN, sample_rate=CSV_SOURCE_HZ,
)
trajectory = raw_trajectory
if CSV_ZERO_START:
    trajectory = Trajectory(raw_trajectory.times - raw_trajectory.times[0],
                            raw_trajectory.positions, raw_trajectory.joint_names)
if trajectory.times[0] != 0:
    raise ValueError("Для отдельного RRT-подвода CSV должен начинаться с t=0; включите CSV_ZERO_START.")
if CSV_RESAMPLE_HZ is not None:
    trajectory = trajectory.resampled(CSV_RESAMPLE_HZ, mode=CSV_RESAMPLE_MODE)
trajectory = trajectory.time_scaled(TIME_SCALE)
print(f"CSV: {len(raw_trajectory.times)} точек → {len(trajectory.times)} точек; "
      f"горизонт {trajectory.times[0]:.6f}…{trajectory.times[-1]:.6f} с")
if len(trajectory.times) > 1:
    dt = np.diff(trajectory.times)
    print(f"Шаги точек: {dt.min():.6f}…{dt.max():.6f} с; ServoJ: {COMMAND_HZ:g} Гц")
endpoints = [(LEFT_IP, LEFT_PORT), (RIGHT_IP, RIGHT_PORT)][:2 if DUAL else 1]
arms = [Arm(name=("left", "right")[i], ip=ip, port=port,
            joint_names=scene.joint_names[i*6:(i+1)*6], enable_motion=True,
            command_rate=COMMAND_HZ, read_rate=READ_HZ, advance_ms=ADVANCE_MS[i],
            **(LEFT_LIMITS, RIGHT_LIMITS)[i])
        for i, (ip, port) in enumerate(endpoints)]

## Предпросмотр до подключения: углы, скорости, ускорения и лимиты
`interpolate` создаёт точную сетку (кроме возможного короткого последнего интервала). `select` выбирает ближайшие исходные строки с их исходным временем: 100 → 30 Гц даёт шаги 30/40 мс, не точные 33.333 мс. Обе конечные точки сохраняются; фильтрации нет, удаление точек может изменить путь.
Замедление выполняется **после** выбора точек: при TIME_SCALE=n скорость уменьшается в n раз, оценка ускорения — в n² раз. 30 Гц при n=3 становятся 10 Гц по точкам, а ServoJ остаётся COMMAND_HZ.
Для positions-only CSV скорость берётся по линейным сегментам, ускорение — численная оценка изменения скоростей соседних сегментов. В узлах/start/end возможны скачки скорости: конечный график ускорений не доказывает плавность. Эти производные **не передаются** контроллеру. Линии лимитов учитывают URDF; max_step остаётся отдельной защитой на каждую команду.
При текущих консервативных лимитах пример CSV на исходной скорости превышает численную оценку ускорения; TIME_SCALE=2 устраняет эти превышения. Не увеличивайте аппаратные лимиты только ради прохождения проверки.


In [ ]:
figures, limit_report = plot_trajectory(trajectory, arms, scene=scene)
for name, report in limit_report.items():
    print(name, "превышения:", report["violations"],
          "ускорение:", report["acceleration_kind"])


In [ ]:
# Только эта ячейка подключает контроллер; preview выше не обращается к роботу.
mocks = []
if USE_MOCK:
    for i, arm in enumerate(arms):
        mock = MockRobot(positions=trajectory.positions[0, i*6:(i+1)*6].tolist()).start()
        mocks.append(mock)
        arm.ip, arm.port = mock.host, mock.port
robot = Robot(arms, scene=scene).connect(viewer=VIEWER)
robot.get_state()


## 1. Отдельный подвод к первой точке CSV через RRTConnect

In [ ]:
approach_execution = None
if RUN_APPROACH:
    try:
        approach = robot.plan(trajectory.positions[0], scale=.2)
        approach_execution = robot.execute_trajectory(approach, start_delay=START_DELAY_S)
        print(approach_execution.wait(approach.times[-1] + START_DELAY_S + 10))
    except Exception as error:
        print(type(error).__name__, error)
    finally:
        if approach_execution is not None:
            figures, metrics = plot_execution(approach_execution, OUTPUT / "approach")
            print(metrics)
else:
    print("Для подвода установите RUN_APPROACH=True и повторите эту ячейку.")

## 2. Выполнить подготовленный CSV
Нет entry/lead-out и автоматически добавленных производных. Применяются только явно выбранные resampling и TIME_SCALE. Обе руки получают один `start_unix_ns`. Перед отправкой проверяются позиции, скорости сегментов и численная оценка ускорения; превышения блокируют отправку CSV. Это не гарантия непрерывного ограничения ускорения. Collision preflight дискретный. После изменения лимитов переподключите robot: параметры контроллера задаются при connect.


In [ ]:
csv_execution = None
if RUN_CSV:
    try:
        # Свежая проверка текущего CSV и лимитов, без генерации dq/ddq для отправки.
        report = trajectory_limit_report(trajectory, robot.arms, scene=robot.scene)
        violations = {name: r["violations"] for name, r in report.items() if r["has_violations"]}
        if violations:
            raise ValueError(f"CSV превышает лимиты preview: {violations}. "
                             "Увеличьте TIME_SCALE или исправьте траекторию/настройки.")
        csv_execution = robot.execute_trajectory(trajectory, start_delay=START_DELAY_S)
        print(csv_execution.wait(trajectory.times[-1] + START_DELAY_S + 10))
    except Exception as error:
        print(type(error).__name__, error)
    finally:
        if csv_execution is not None:
            figures, metrics = plot_execution(csv_execution, OUTPUT / "csv")
            print(metrics)
        print(robot.diagnostics())
else:
    print("Для выполнения установите RUN_CSV=True и повторите эту ячейку.")

## Дополнительно: случайная суставная цель, RRTConnect и явное выполнение

In [ ]:
RUN_RANDOM = False
random_execution = None
if RUN_RANDOM:
    try:
        measured = np.concatenate([arm["positions"] for arm in robot.get_state()["arms"]])
        target = measured + np.random.default_rng().uniform(-.15, .15, len(measured))
        random_plan = robot.plan(target, scale=.2)  # невалидная цель отвергается.
        random_execution = robot.execute_trajectory(random_plan, start_delay=START_DELAY_S)
        print(random_execution.wait(random_plan.times[-1] + START_DELAY_S + 10))
    except Exception as error:
        print(type(error).__name__, error)
    finally:
        if random_execution is not None:
            plot_execution(random_execution, OUTPUT / "random")

In [ ]:
# robot.cancel() — отдельная явная отмена активного выполнения.
robot.close()
for mock in mocks:
    mock.close()